# Generate and explore r0 datasets

Issue #27: linear, grouped, polynomial, quadratic, Fourier, periodic, smooth, exact GP and composed tasks. Create datasets on CPU or GPU, save resumable groups to Drive, and use the embedded explorer to inspect charts, rows, parameters and complete exports.

Run cells in order. Select a GPU runtime before choosing `cuda`. Installation requires Python 3.12+. The notebook uses the generator's NumPy/Torch dependencies, not the evaluation stack. Restart the runtime after installation if those libraries or r0 were already imported. Committed outputs are empty; hosted acceptance must be established by executing this notebook in Colab. Generation graphs, response curves and paired views are still being implemented.

In [ ]:
import subprocess
import sys

assert sys.version_info >= (3, 12), "Select a Python 3.12+ runtime"
REVISION = "8794ad9581eafa3fa17ef77d279613058512acdc"
subprocess.run(
    [sys.executable, "-m", "pip", "install", "numpy==2.5.5", "torch==2.15.0"], check=True
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-deps",
        "--force-reinstall",
        f"git+https://github.com/Avistian/r0.git@{REVISION}",
    ],
    check=True,
)

## Persistent output
Mount Drive once. Use one notebook coordinator per output directory; Drive does not provide a distributed writer lock. Keep the same recipe, seed and limits to resume. Change the output directory when changing a request. Interrupted groups stay incomplete; failed tasks require `retry_failed=True` explicitly.


In [ ]:
from pathlib import Path

USE_DRIVE = True  # False uses temporary Colab storage, lost when the runtime is removed.
if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/r0-generation")
else:
    ROOT = Path("/content/r0-generation")
ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
import json
from dataclasses import asdict, replace

import torch

from r0.modules.explorer import ExplorerSession
from r0.modules.generation import RunRequest, run_guarded
from r0.modules.priors import recipe_from_dict, record_fingerprint

DEVICE = "cpu"  # Change to "cuda" after selecting a GPU runtime.
assert DEVICE in ("cpu", "cuda")
assert DEVICE != "cuda" or torch.cuda.is_available(), "CUDA unavailable; no fallback"
print(
    {
        "source": REVISION,
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
        "device": torch.cuda.get_device_name() if DEVICE == "cuda" else "cpu",
    }
)
examples = ExplorerSession(ROOT).request("examples", {})["examples"]
print("Available families:", list(examples))
FAMILY = "Linear"  # Select any exact name printed above.
recipe_config = examples[FAMILY]
controls = recipe_config.get("task", recipe_config)
controls.update(rows=128, context_rows=96, noise_ratio=0.5)
recipe = recipe_from_dict(recipe_config)
print(json.dumps(recipe_config, indent=2))
smoke_request = RunRequest(
    recipe, seed=27, count=8, group_size=4, max_seconds=60, max_bytes=10 * 1024**2, device=DEVICE
)
smoke_key = record_fingerprint({"request": asdict(smoke_request), "source": REVISION})
smoke_dir = ROOT / "smoke" / smoke_key
smoke = run_guarded(smoke_request, smoke_dir, workers=2)
print(json.dumps(asdict(smoke), indent=2))
print("Saved output:", smoke_dir)
assert smoke.returncode == 0, smoke.stderr
assert not smoke.timed_out, smoke.stderr

## Estimate before scaling
The report's `generated` count includes successful computations, including any that could not be committed. `completed` counts all verified saved tasks, including earlier invocations. Time includes generation, serialization and persistence, but excludes interpreter startup; `elapsed_seconds` includes startup. Peak RSS is the child process lifetime peak; GPU memory is PyTorch's allocated peak, not total device memory. A killed child has no completed report: do not infer zero failures or memory use.

A fresh smoke run retains its measurement next to its output directory. On resume the notebook loads that measurement and labels it historical; it does not treat skipped tasks as a new timing. If hardware changes, use a fresh smoke directory and measurement before estimating new workloads. Estimates apply only to this recipe/hardware and are not measured 100M-task performance.


In [ ]:
COUNT = 1000  # Explicitly choose the desired task count, up to 100_000_000.
MAX_SECONDS = 300
MAX_BYTES = 100 * 1024**2
WORKERS = 2
summary = smoke.summary
measurement_path = smoke_dir.with_suffix(".measurement.json")
identity = json.loads(json.dumps({"request": asdict(smoke_request), "source": REVISION}))
if summary["generated"] > 0:
    measurement_path.write_text(json.dumps({"identity": identity, "summary": summary}))
else:
    saved = json.loads(measurement_path.read_text())
    assert saved["identity"] == identity, "Measurement belongs to a different request"
    summary = saved["summary"]
    print("Using retained measurement from earlier smoke; not a new timing:", summary)
rate = summary["generated_per_second"]
print(
    {
        "measured_tasks_per_second": rate,
        "estimated_seconds": COUNT / rate,
        "estimate_scope": "same recipe and hardware; storage and larger-run overhead may differ",
    }
)
large_request = replace(smoke_request, count=COUNT, max_seconds=MAX_SECONDS, max_bytes=MAX_BYTES)
print(json.dumps(asdict(large_request), indent=2))

In [ ]:
RUN_LARGER = False  # Set True after reviewing the estimate and limits above.
if RUN_LARGER:
    large_key = record_fingerprint({"request": asdict(large_request), "source": REVISION})
    large = run_guarded(
        large_request, ROOT / "large" / large_key, workers=WORKERS, retry_failed=False
    )
    print(json.dumps(asdict(large), indent=2))

## Verify and export
ZIP groups contain lossless predictor/truth arrays and JSON cards. Hidden truth is separate from predictor inputs. Verify each completed archive before using it. Drive keeps the groups for resume or download; downloading below exports one bounded group, not the entire large run. Throughput and Colab durability still need measurement on your selected runtime.


In [ ]:
from r0.modules.generation import verify_group

archives = sorted(smoke_dir.glob("groups/*/*/data-*.zip"))
for archive in archives:
    if archive.with_suffix(".json").exists():
        print(archive.relative_to(ROOT), verify_group(archive))
# Optional: from google.colab import files; files.download(str(archives[0]))

## Embedded dataset explorer
Run the next cell in this Colab session. Its controls and charts are the same as the local explorer; generation happens on this hosted kernel through the shared bounded runner. No local server or separate website is needed.

Choose a family and generate a task, or open a saved archive using the relative path printed above and a task identifier in its receipt. The explorer has its own explicit CPU/CUDA selector under **Recipe and run limits**. Its one-task generation requests are independent of the bulk smoke settings. Drive outputs and complete replay bundles keep predictor inputs separate from hidden truth.

Re-running this cell unregisters the previous callback and releases its dataset. Old controls then become inactive. After reconnecting to a new runtime, rerun installation/import/setup cells before using the UI; a saved notebook picture is not a live explorer.

In [ ]:
from contextlib import suppress

from google.colab import output

from r0.modules.explorer import show_colab_explorer

previous_callback = globals().pop("explorer_callback", None)
if previous_callback is not None:
    with suppress(ValueError):
        output.unregister_callback(previous_callback)
explorer_callback = show_colab_explorer(ROOT)

## Check the hosted runtime
Local transport tests do not establish Colab acceptance. In this hosted notebook, check:

1. Generate a regression task and inspect its distributions, signal and an individual row.
2. Switch to multiclass and confirm probabilities and context/query class counts.
3. Replay the recipe and confirm zero new generations in run measurements.
4. Download the complete dataset ZIP and an SVG/PNG chart; reopen the saved archive.
5. If a GPU runtime is selected, choose CUDA in the explorer and repeat generation. Confirm the displayed execution record names the actual GPU.
6. Reconnect the runtime and verify a completed Drive group before resuming it.

Record the source revision, actual hardware, run measurements, failures and observed UI behavior. Kernel generation checks, Drive persistence, browser interaction and CPU/GPU agreement are separate evidence. Do not mark hosted acceptance complete from the notebook's presence or from local emulation.